<a href="https://colab.research.google.com/github/ryouchinsa/Rectlabel-support/blob/master/notebooks/train_rf_detr_object_detection_on_custom_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Train a RF-DETR Object Detetion Model with Custom Data

We will show you how to train a RF-DETR object detection model with your images and annotations and export to a Core ML model which can be used for auto labeling on RectLabel.

### Use GPU

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Runtime` -> `Change runtime type` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

Wed Apr  1 17:46:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install PyTorch 2.8.0

In [ ]:
!pip install -q torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 94.5 MB/s eta 0:00:00


### Install RF-DETR



In [ ]:
!pip install -q rfdetr[train,loggers]==1.6.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

### Download training images and annotations

Download training images and annotations. You can use these or replace them with your own data.

In [ ]:
!mkdir datasets
%cd datasets
!wget -q https://huggingface.co/datasets/rectlabel/datasets/resolve/main/converse_vans_detection_coco.zip
!unzip -q converse_vans_detection_coco.zip
%cd ..

/content/datasets
/content


### Fine-tune RF-DETR on custom dataset

Start training from the current content folder.

In [ ]:
from rfdetr import RFDETRNano

model = RFDETRNano(num_classes=2)
dataset_dir = "datasets/converse_vans_detection_coco"
model.train(dataset_dir=dataset_dir, epochs=10, batch_size=4, grad_accum_steps=4)

[2026-04-01 18:01:09] [INFO] rf-detr - File rf-detr-nano.pth already exists with correct MD5 hash.


[2026-04-01 18:01:09] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-01 18:01:09] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-04-01 18:01:10] [INFO] rf-detr - File rf-detr-nano.pth already exists with correct MD5 hash.


[2026-04-01 18:01:11] [WARNING] rf-detr - Checkpoint has 90 classes but model is configured for 2. The detection head will be re-initialized to 2 classes.
[2026-04-01 18:01:12] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-01 18:01:12] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-04-01 18:01:13] [INFO] rf-detr - File rf-detr-nano.pth already exists with correct MD5 hash.


...

Trainable params: 30.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 30.2 M                                                                                               
Total estimated model params size (MB): 120                                                                        
Modules in train mode: 449                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

[2026-04-01 18:01:17] [INFO] rf-detr - Best EMA mAP improved to 0.5077 (epoch 0)
[2026-04-01 18:02:15] [INFO] rf-detr - Best regular mAP saved to /content/output/checkpoint_best_regular.pth (epoch 0)
[2026-04-01 18:03:14] [INFO] rf-detr - Best regular mAP saved to /content/output/checkpoint_best_regular.pth (epoch 1)
[2026-04-01 18:05:03] [INFO] rf-detr - Best EMA mAP improved to 0.5202 (epoch 3)
[2026-04-01 18:05:58] [INFO] rf-detr - Best EMA mAP improved to 0.5885 (epoch 4)
[2026-04-01 18:05:59] [INFO] rf-detr - Best regular mAP saved to /content/output/checkpoint_best_regular.pth (epoch 4)
[2026-04-01 18:06:55] [INFO] rf-detr - Best regular mAP saved to /content/output/checkpoint_best_regular.pth (epoch 5)
[2026-04-01 18:07:52] [INFO] rf-detr - Best EMA mAP improved to 0.6091 (epoch 6)
[2026-04-01 18:07:52] [INFO] rf-detr - Best regular mAP saved to /content/output/checkpoint_best_regular.pth (epoch 6)
[2026-04-01 18:08:47] [INFO] rf-detr - Best EMA mAP improved to 0.6386 (epoch 7)


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


[2026-04-01 18:10:44] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.6430, ema=0.6493)


The trained model is checkpoint_best_total.pth.

In [ ]:
!ls -la /content/output

total 353984
drwxr-xr-x 2 root root      4096 Apr  1 18:10 .
drwxr-xr-x 1 root root      4096 Apr  1 17:50 ..
-rw-r--r-- 1 root root 120805617 Apr  1 18:09 checkpoint_best_ema.pth
-rw-r--r-- 1 root root 120807565 Apr  1 18:09 checkpoint_best_regular.pth
-rw------- 1 root root 120795321 Apr  1 18:10 checkpoint_best_total.pth
-rw-r--r-- 1 root root     16357 Apr  1 17:59 events.out.tfevents.1775065829.3497f7994c9a.9807.0
-rw-r--r-- 1 root root     16357 Apr  1 18:10 events.out.tfevents.1775066474.3497f7994c9a.9807.1
-rw-r--r-- 1 root root         3 Apr  1 17:50 hparams.yaml
-rw-r--r-- 1 root root      6470 Apr  1 18:10 metrics.csv


Before exporting to a Core ML model, edit a line of transformer.py.

In [ ]:
!pip show rfdetr

Name: rfdetr
Version: 1.6.0
Summary: RF-DETR
Home-page: https://github.com/roboflow/rf-detr
Author: 
Author-email: "Roboflow, Inc" <develop@roboflow.com>
License: Apache License 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: peft, pydantic, pyDeprecate, requests, supervision, torch, torchvision, tqdm, transformers
Required-by: 


In [ ]:
path = "/usr/local/lib/python3.12/dist-packages/rfdetr/models/transformer.py"
with open(path, "r") as f:
    content = f.read()
modified_content = content.replace("mask_flatten, spatial_shapes_hw", "mask_flatten, spatial_shapes")
with open(path, "w") as f:
    f.write(modified_content)

### Install RF-DETR to CoreML

In [ ]:
!git clone https://github.com/ryouchinsa/rf-detr-to-coreml.git
%cd rf-detr-to-coreml
!pip install -q -e .

Cloning into 'rf-detr-to-coreml'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 109 (delta 44), reused 89 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 1.72 MiB | 8.71 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/rf-detr-to-coreml
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
  Building editable for rfdetr-coreml (pyproject.toml) ... done


Move the best model to the current folder and export to a Core ML model.

In [ ]:
!mv /content/output/checkpoint_best_total.pth .

In [ ]:
!python export_coreml.py --model nano --weights checkpoint_best_total.pth

scikit-learn version 1.6.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
XGBoost version 3.2.0 has not been tested with coremltools. You may run into unexpected errors. XGBoost 1.4.2 is the most recent version that has been tested.
2026-04-01 18:11:32.122248: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775067092.143273   16244 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775067092.150276   16244 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775067092.167844   16244 computation_placer.cc:177] computation placer already registered. Please check linkage and 

In [ ]:
!ls -la output

total 12
drwxr-xr-x 3 root root 4096 Apr  1 18:11 .
drwxr-xr-x 7 root root 4096 Apr  1 18:11 ..
drwxr-xr-x 3 root root 4096 Apr  1 18:11 rf-detr-nano-checkpoint_best_total-fp32.mlpackage


In [ ]:
%cd output

/content/rf-detr-to-coreml/output


Zip the Core ML model and download it from the File browser at the left hand. You can auto label images using the Core ML model on RectLabel.

In [ ]:
!zip -r nano.zip rf-detr-nano-checkpoint_best_total-fp32.mlpackage

  adding: rf-detr-nano-checkpoint_best_total-fp32.mlpackage/ (stored 0%)
  adding: rf-detr-nano-checkpoint_best_total-fp32.mlpackage/Manifest.json (deflated 59%)
  adding: rf-detr-nano-checkpoint_best_total-fp32.mlpackage/Data/ (stored 0%)
  adding: rf-detr-nano-checkpoint_best_total-fp32.mlpackage/Data/com.apple.CoreML/ (stored 0%)
  adding: rf-detr-nano-checkpoint_best_total-fp32.mlpackage/Data/com.apple.CoreML/weights/ (stored 0%)
  adding: rf-detr-nano-checkpoint_best_total-fp32.mlpackage/Data/com.apple.CoreML/weights/weight.bin (deflated 7%)
  adding: rf-detr-nano-checkpoint_best_total-fp32.mlpackage/Data/com.apple.CoreML/model.mlmodel (deflated 89%)
